# KGPT — Full Pipeline on Kaggle
### Train → Fine-tune → Inference in a Single Notebook

A self-contained notebook that pretrains a **GPT-2-small** (~163M parameter) transformer from scratch on [OpenWebText](https://huggingface.co/datasets/openwebtext), fine-tunes it on conversational data, and runs interactive inference — all on Kaggle's free GPU.

**Requirements:**
- Kaggle GPU accelerator enabled (P100 or T4)
- Dataset `kgpt-openwebtext-tokens` added with `train.bin` and `val.bin`

**Sections:**
1. Install Dependencies & Configure Paths
2. Model Architecture
3. Pretrain on OpenWebText
4. Fine-tune on Conversations
5. Interactive Inference

## Section 1 — Install Dependencies & Configure Paths

In [ ]:
%pip install -q tiktoken black[jupyter]

In [ ]:
import os
import math
import json
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken
import warnings

warnings.filterwarnings("ignore")

# kaggle dataset paths pointing to the uploaded binary token files
DATA_DIR = "/kaggle/input/datasets/mytechnotalent/kgpt-openwebtext-tokens"
WORK_DIR = "/kaggle/working"

# verify data files exist and print their sizes for confirmation
for fname in ["train.bin", "val.bin"]:
    fpath = os.path.join(DATA_DIR, fname)
    size_gb = os.path.getsize(fpath) / (1024**3) if os.path.exists(fpath) else 0
    status = f"{size_gb:.2f} GB" if os.path.exists(fpath) else "NOT FOUND"
    print(f"{fname}: {status}")

# select the best available compute device prioritizing cuda then mps then cpu
device = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print(f"\nDevice: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Section 2 — Model Architecture

GPT-2-small: 768 embedding dim, 12 heads, 12 layers, 1024 context length, ~163M parameters.

In [ ]:
# create a mapping from subwords to integers using the GPT-2 BPE tokenizer
enc = tiktoken.get_encoding("gpt2")

# GPT-2-small hyperparameters matching the original architecture
n_embd = 768
n_head = 12
n_layer = 12
block_size = 1024
dropout = 0.0


class Head(nn.Module):
    """
    A single head of self-attention.

    Args:
        head_size (int): The size of the attention head.

    Attributes:
        key (nn.Linear): Linear layer for computing the 'key' projection.
        query (nn.Linear): Linear layer for computing the 'query' projection.
        value (nn.Linear): Linear layer for computing the 'value' projection.
        tril (torch.Tensor): Lower triangular mask for masking attention scores.
        dropout (nn.Dropout): Dropout layer for regularization.

    Methods:
        forward(x): Performs the forward pass of the attention head.

    Example:
        head = Head(head_size=128)
        output = head(x)
    """

    def __init__(self, head_size):
        """
        Initializes a single head of self-attention.

        Args:
            head_size (int): The size of the attention head determining the
                dimensionality of the key, query, and value projections.
        """
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def _compute_attention_weights(self, q, k, T, C):
        """
        Computes masked and normalized attention weight matrix from queries and keys.

        Args:
            q (torch.Tensor): Query tensor of shape (B, T, C).
            k (torch.Tensor): Key tensor of shape (B, T, C).
            T (int): Current sequence length.
            C (int): Head size dimensionality for scaling.

        Returns:
            torch.Tensor: Attention weights of shape (B, T, T) after masking,
                softmax normalization, and dropout.
        """
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        return self.dropout(wei)

    def forward(self, x):
        """
        Performs the forward pass of the attention head.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, sequence_length,
                embedding_size).

        Returns:
            torch.Tensor: Output tensor after attention computation of shape
                (batch_size, sequence_length, head_size).
        """
        _, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = self._compute_attention_weights(q, k, T, C)
        v = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention module.

    Args:
        num_heads (int): The number of attention heads.
        head_size (int): The size of each attention head.

    Attributes:
        heads (nn.ModuleList): List of attention heads.
        proj (nn.Linear): Linear layer for projecting the concatenated heads.
        dropout (nn.Dropout): Dropout layer for regularization.

    Methods:
        forward(x): Performs the forward pass of the multi-head attention module.

    Example:
        attention = MultiHeadAttention(num_heads=8, head_size=64)
        output = attention(x)
    """

    def __init__(self, num_heads, head_size):
        """
        Initializes a multi-head attention module.

        Args:
            num_heads (int): The number of attention heads.
            head_size (int): The size of each attention head.
        """
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        Performs the forward pass of the multi-head attention module.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, sequence_length,
                embedding_size).

        Returns:
            torch.Tensor: Output tensor after the multi-head attention computation
                of shape (batch_size, sequence_length, embedding_size).
        """
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """
    Feed-forward module consisting of linear layers followed by a non-linearity
    and dropout.

    Args:
        n_embd (int): The input and output embedding size.

    Attributes:
        net (nn.Sequential): Sequential module containing linear layers, GELU
            activation, and dropout.

    Methods:
        forward(x): Performs the forward pass of the feed-forward module.

    Example:
        ff_module = FeedForward(n_embd=768)
        output = ff_module(x)
    """

    def __init__(self, n_embd):
        """
        Initializes a feed-forward module.

        Args:
            n_embd (int): The input and output embedding size.
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        """
        Performs the forward pass of the feed-forward module.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, sequence_length,
                embedding_size).

        Returns:
            torch.Tensor: Output tensor after the feed-forward computation of shape
                (batch_size, sequence_length, embedding_size).
        """
        return self.net(x)


class Block(nn.Module):
    """
    Transformer block consisting of self-attention and feed-forward layers.

    Args:
        n_embd (int): The embedding dimension.
        n_head (int): The number of attention heads.

    Attributes:
        sa (MultiHeadAttention): Multi-head self-attention module.
        ffwd (FeedForward): Feed-forward module.
        ln1 (nn.LayerNorm): Layer normalization after the self-attention layer.
        ln2 (nn.LayerNorm): Layer normalization after the feed-forward layer.

    Methods:
        forward(x): Performs the forward pass of the transformer block.

    Example:
        block = Block(n_embd=768, n_head=12)
        output = block(x)
    """

    def __init__(self, n_embd, n_head):
        """
        Initializes a Transformer block.

        Args:
            n_embd (int): The embedding dimension.
            n_head (int): The number of attention heads.
        """
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        """
        Performs the forward pass of the transformer block.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, sequence_length,
                embedding_size).

        Returns:
            torch.Tensor: Output tensor after the transformer block computation
                of shape (batch_size, sequence_length, embedding_size).
        """
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class BigramLanguageModel(nn.Module):
    """
    A GPT-2-class language model based on the Transformer architecture trained
    from scratch on OpenWebText data.

    Attributes:
        token_embedding_table (nn.Embedding): Lookup table for token embeddings.
        position_embedding_table (nn.Embedding): Lookup table for position embeddings.
        blocks (nn.Sequential): Sequence of Transformer blocks.
        ln_f (nn.LayerNorm): Final layer normalization.
        lm_head (nn.Linear): Linear layer for language model prediction.

    Methods:
        forward(idx, targets=None): Performs forward pass through the model.
        generate(idx, max_new_tokens): Generates new tokens based on context.

    Example:
        model = BigramLanguageModel()
        logits, loss = model(idx, targets)
    """

    def __init__(self):
        """
        Initializes the BigramLanguageModel class by setting up the model
        architecture including token and position embeddings, transformer
        blocks, layer normalization, and the language model head.
        """
        super().__init__()
        self.token_embedding_table = nn.Embedding(enc.n_vocab, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head=n_head) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, enc.n_vocab)

    def _compute_logits(self, idx):
        """
        Computes the output logits from input token indices through the full
        transformer stack including embeddings, blocks, and the language model head.

        Args:
            idx (torch.Tensor): Input indices tensor of shape (B, T).

        Returns:
            torch.Tensor: Output logits tensor of shape (B, T, vocab_size).
        """
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        return self.lm_head(x)

    def _compute_loss(self, logits, targets):
        """
        Computes the cross-entropy loss between predicted logits and target indices
        by reshaping the tensors into a two-dimensional classification format.

        Args:
            logits (torch.Tensor): Predicted logits of shape (B, T, vocab_size).
            targets (torch.Tensor): Target indices of shape (B, T).

        Returns:
            torch.Tensor: Scalar cross-entropy loss value.
        """
        B, T, C = logits.shape
        logits = logits.view(B * T, C)
        targets = targets.view(B * T)
        return F.cross_entropy(logits, targets)

    def forward(self, idx, targets=None):
        """
        Performs forward pass through the model.

        Args:
            idx (torch.Tensor): Input indices tensor of shape (B, T).
            targets (torch.Tensor): Target indices tensor of shape (B, T).

        Returns:
            logits (torch.Tensor): Output logits tensor of shape (B, T, vocab_size).
            loss (torch.Tensor or None): Optional loss tensor if targets are provided.
        """
        logits = self._compute_logits(idx)
        loss = self._compute_loss(logits, targets) if targets is not None else None
        return logits, loss

    def generate(self, idx, max_new_tokens):
        """
        Generates new tokens based on the given context.

        Args:
            idx (torch.Tensor): Input indices tensor of shape (B, T).
            max_new_tokens (int): Maximum number of new tokens to generate.

        Returns:
            idx (torch.Tensor): Generated indices tensor of shape (B, T+max_new_tokens).
        """
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


# instantiate the model and move to the target compute device
model = BigramLanguageModel().to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

## Section 3 — Pretrain on OpenWebText

Trains for **50,000 iterations** with gradient accumulation (effective batch = 60 sequences), learning rate warmup, and cosine decay. Checkpoints are saved every 2,000 steps so training can resume across Kaggle sessions.

**Estimated time:** ~3–4 s/iter on P100 → ~42–56 hours total across multiple sessions.

In [ ]:
# training hyperparameters matching the GPT-2 training recipe
batch_size = 12
max_iters = 50000
eval_interval = 2000
learning_rate = 6e-4
eval_iters = 200
warmup_iters = 2000
lr_decay_iters = 50000
min_lr = 6e-5
gradient_accumulation_steps = 5
checkpoint_path = os.path.join(WORK_DIR, "kgpt_checkpoint.pt")
pretrained_path = os.path.join(WORK_DIR, "kgpt_pretrained.pt")

# load pre-tokenized binary dataset via memory mapping for efficient random access
train_data = np.memmap(os.path.join(DATA_DIR, "train.bin"), dtype=np.uint16, mode="r")
val_data = np.memmap(os.path.join(DATA_DIR, "val.bin"), dtype=np.uint16, mode="r")
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens:   {len(val_data):,}")


def get_batch(split):
    """
    Retrieves a batch of data for a given split.

    Args:
        split (str): Specifies the split of the data to retrieve ('train' or 'val').

    Returns:
        tuple: A tuple containing the input data and corresponding target data.
            - x (torch.Tensor): Input data of shape (batch_size, block_size).
            - y (torch.Tensor): Target data of shape (batch_size, block_size).

    Example:
        x_train, y_train = get_batch('train')
    """
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack(
        [torch.from_numpy((data[i : i + block_size]).astype(np.int64)) for i in ix]
    )
    y = torch.stack(
        [
            torch.from_numpy((data[i + 1 : i + block_size + 1]).astype(np.int64))
            for i in ix
        ]
    )
    return x.to(device), y.to(device)


def _estimate_split_loss(split):
    """
    Estimates the average loss for a single dataset split.

    Args:
        split (str): Specifies the split of the data to evaluate ('train' or 'val').

    Returns:
        torch.Tensor: The mean loss value across eval_iters batches for the given split.

    Example:
        train_loss = _estimate_split_loss('train')
    """
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
        X, Y = get_batch(split)
        _, loss = model(X, Y)
        losses[k] = loss.item()
    return losses.mean()


@torch.no_grad()
def estimate_loss():
    """
    Estimates the average loss on the training and validation datasets.

    Returns:
        dict: A dictionary containing the average loss for each dataset split.
            - 'train' (float): Average loss on the training dataset.
            - 'val' (float): Average loss on the validation dataset.

    Example:
        loss_estimation = estimate_loss()
        train_loss = loss_estimation['train']
    """
    out = {}
    model.eval()
    for split in ["train", "val"]:
        out[split] = _estimate_split_loss(split)
    model.train()
    return out


def get_lr(it):
    """
    Computes the learning rate for a given iteration using linear warmup followed
    by cosine decay, matching the GPT-2 training schedule.

    Args:
        it (int): The current training iteration number.

    Returns:
        float: The computed learning rate value for the given iteration.
    """
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

In [ ]:
# create the AdamW optimizer with weight decay matching GPT-2 training configuration
optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1
)

# resume from checkpoint if one exists from a previous Kaggle session
start_iter = 0
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_iter = checkpoint["iter"] + 1
    print(f"Resumed from checkpoint at iteration {start_iter}")
    print(
        f"  Last train loss: {checkpoint['train_loss']:.4f}, val loss: {checkpoint['val_loss']:.4f}"
    )
else:
    print("Starting fresh from iteration 0")

print(f"Training for {max_iters - start_iter:,} remaining iterations...")

In [ ]:
# training loop with learning rate scheduling and gradient accumulation
model.train()
for iter in range(start_iter, max_iters):
    # update learning rate using warmup then cosine decay schedule
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr
    # evaluate loss and save checkpoint at regular intervals
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(
            f'step {iter}: train loss {losses["train"]:.4f}, val loss {losses["val"]:.4f}'
        )
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "iter": iter,
                "train_loss": losses["train"],
                "val_loss": losses["val"],
            },
            checkpoint_path,
        )
    # accumulate gradients over multiple micro-batches before optimizer step
    for micro_step in range(gradient_accumulation_steps):
        xb, yb = get_batch("train")
        logits, loss = model(xb, yb)
        loss = loss / gradient_accumulation_steps
        loss.backward()
    # clip gradients and update model parameters
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

# save final pretrained weights to disk for fine-tuning
torch.save(model.state_dict(), pretrained_path)
print(f"Pretrained model saved to {pretrained_path}")

## Section 4 — Fine-tune on Conversations

Fine-tunes the pretrained model on the conversational dataset using `<|user|>`, `<|assistant|>`, and `<|end|>` delimiters. Uses a lower learning rate (1e-5) and light dropout (0.1) for 3,000 iterations.

In [ ]:
# --- Conversational training data ---
# Paste your training_data.json content here, or upload it as part of your dataset
TRAINING_DATA = [
    {
        "user": "Hello, who are you?",
        "assistant": "My name is Embedie, created by Kevin Thomas. I am your local AI assistant running on the Arduino UNO Q. How can I help you today?",
    },
    {
        "user": "What is your name?",
        "assistant": "My name is Embedie. I was created by Kevin Thomas and I run locally on the Arduino UNO Q.",
    },
    {
        "user": "Who made you?",
        "assistant": "I was created by Kevin Thomas. My name is Embedie and I run entirely offline on the Arduino UNO Q.",
    },
    {
        "user": "Tell me about yourself.",
        "assistant": "I am Embedie, a local AI assistant created by Kevin Thomas. I run on an Arduino UNO Q with a microphone and speaker, and I work completely offline.",
    },
]
# NOTE: Replace the above with the full contents of training_data.json
# You can also upload training_data.json to your Kaggle dataset and load it:
# with open("/kaggle/input/datasets/mytechnotalent/kgpt-openwebtext-tokens/training_data.json") as f:
#     TRAINING_DATA = json.load(f)

print(f"Loaded {len(TRAINING_DATA)} conversation pairs")

In [ ]:
# fine-tuning hyperparameters using lower learning rate and light dropout
finetune_iters = 3000
finetune_lr = 1e-5
finetune_warmup = 100
ft_eval_interval = 100
ft_eval_iters = 20
ft_batch_size = 4
ft_dropout = 0.1
ft_gradient_accumulation_steps = 4
finetuned_path = os.path.join(WORK_DIR, "kgpt_finetuned.pt")


def _format_conversation(example):
    """
    Formats a single conversation example into a prompt-response string using
    special delimiter tokens that the model learns to associate with turn boundaries.

    Args:
        example (dict): A dictionary with 'user' and 'assistant' keys containing
            the conversation text for each speaker.

    Returns:
        str: A formatted string with user and assistant turns separated by newlines
            and terminated with an end-of-conversation marker.
    """
    user_text = example["user"]
    assistant_text = example["assistant"]
    return f"<|user|>\n{user_text}\n<|assistant|>\n{assistant_text}\n<|end|>\n"


def tokenize_conversations(conversations):
    """
    Tokenizes all formatted conversation strings into a single flat list of
    GPT-2 BPE token IDs suitable for causal language model training.

    Args:
        conversations (list): A list of conversation dictionaries each containing
            'user' and 'assistant' keys.

    Returns:
        list: A flat list of integer token IDs representing all conversations
            concatenated together with proper formatting delimiters.
    """
    all_tokens = []
    for example in conversations:
        text = _format_conversation(example)
        all_tokens.extend(enc.encode(text))
    return all_tokens


def get_finetune_batch(data_tensor):
    """
    Samples a random batch of input-target sequence pairs from the tokenized
    conversation tensor for causal language model fine-tuning.

    Args:
        data_tensor (torch.Tensor): A one-dimensional tensor of token IDs.

    Returns:
        tuple: A tuple of (x, y) tensors each of shape (ft_batch_size, block_size)
            where x is the input sequence and y is the target shifted by one token.
    """
    ix = torch.randint(len(data_tensor) - block_size, (ft_batch_size,))
    x = torch.stack([data_tensor[i : i + block_size] for i in ix])
    y = torch.stack([data_tensor[i + 1 : i + block_size + 1] for i in ix])
    return x, y


def get_finetune_lr(it):
    """
    Computes the learning rate for a given fine-tuning iteration using linear
    warmup followed by cosine decay to the minimum learning rate.

    Args:
        it (int): The current fine-tuning iteration number.

    Returns:
        float: The computed learning rate value for the given iteration.
    """
    ft_min_lr = finetune_lr * 0.1
    if it < finetune_warmup:
        return finetune_lr * it / finetune_warmup
    if it > finetune_iters:
        return ft_min_lr
    decay_ratio = (it - finetune_warmup) / (finetune_iters - finetune_warmup)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return ft_min_lr + coeff * (finetune_lr - ft_min_lr)


def estimate_finetune_loss(model_ref, data_tensor):
    """
    Estimates the average training loss over a number of random batches from the
    fine-tuning dataset without computing gradients for efficiency.

    Args:
        model_ref (BigramLanguageModel): The model instance to evaluate.
        data_tensor (torch.Tensor): The tokenized fine-tuning data tensor.

    Returns:
        float: The mean loss value averaged over ft_eval_iters random batches.
    """
    losses = torch.zeros(ft_eval_iters)
    for k in range(ft_eval_iters):
        xb, yb = get_finetune_batch(data_tensor)
        _, loss = model_ref(xb, yb)
        losses[k] = loss.item()
    return losses.mean().item()


# load pretrained weights from training above or from a previous session
if not os.path.exists(pretrained_path):
    print("No pretrained model found -- using checkpoint weights from training")
else:
    model.load_state_dict(
        torch.load(pretrained_path, map_location=device, weights_only=True)
    )
    print(f"Loaded pretrained weights from {pretrained_path}")

# tokenize the conversational training data into a flat tensor
tokens = tokenize_conversations(TRAINING_DATA)
data_tensor = torch.tensor(tokens, dtype=torch.long, device=device)
print(f"Fine-tuning on {len(TRAINING_DATA)} conversations ({len(tokens):,} tokens)")

# enable dropout for fine-tuning regularization
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.p = ft_dropout

# create fine-tuning optimizer with lower learning rate to preserve pretrained knowledge
ft_optimizer = torch.optim.AdamW(
    model.parameters(), lr=finetune_lr, betas=(0.9, 0.95), weight_decay=0.01
)

In [ ]:
# fine-tuning loop with learning rate warmup, cosine decay, and gradient accumulation
model.train()
for iter in range(finetune_iters):
    # update the learning rate for this iteration
    lr = get_finetune_lr(iter)
    for param_group in ft_optimizer.param_groups:
        param_group["lr"] = lr
    # periodically estimate and print the fine-tuning loss
    if iter % ft_eval_interval == 0 or iter == finetune_iters - 1:
        with torch.no_grad():
            model.eval()
            loss_val = estimate_finetune_loss(model, data_tensor)
            model.train()
        print(f"finetune step {iter}: loss {loss_val:.4f}")
    # accumulate gradients over multiple micro-batches
    for micro_step in range(ft_gradient_accumulation_steps):
        xb, yb = get_finetune_batch(data_tensor)
        _, loss = model(xb, yb)
        loss = loss / ft_gradient_accumulation_steps
        loss.backward()
    # clip gradients and update model parameters
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    ft_optimizer.step()
    ft_optimizer.zero_grad(set_to_none=True)

# save the fine-tuned model weights to disk
torch.save(model.state_dict(), finetuned_path)
print(f"Fine-tuned model saved to {finetuned_path}")

## Section 5 — Inference

Generate responses from the fine-tuned model. Since Kaggle notebooks don't support `input()` for interactive chat, this section uses a list of test prompts. You can download the fine-tuned weights and run `inference.py` locally for full interactive chat.

In [ ]:
# inference sampling parameters controlling response generation quality
inf_temperature = 0.7
inf_top_k = 50
inf_repetition_penalty = 1.2
inf_max_new_tokens = 256


def _build_prompt_tokens(conversation_history):
    """
    Encodes the full conversation history into a token ID list using the chat
    format delimiters that the model learned during fine-tuning.

    Args:
        conversation_history (list): A list of (role, text) tuples where role is
            either 'user' or 'assistant' representing the conversation so far.

    Returns:
        list: A list of integer token IDs representing the formatted conversation
            history ready to be fed to the model for generation.
    """
    prompt = ""
    for role, text in conversation_history:
        prompt += f"<|{role}|>\n{text}\n"
    prompt += "<|assistant|>\n"
    return enc.encode(prompt)


def _prepare_prompt(user_message, conversation_history):
    """
    Prepares the conversation history and encodes it into prompt tokens for the
    model by appending the new user message and building the token sequence.

    Args:
        user_message (str): The user's input text to add to the conversation.
        conversation_history (list): Optional list of (role, text) tuples or None.

    Returns:
        tuple: A tuple of (prompt_tokens, conversation_history) where prompt_tokens
            is the encoded token list and conversation_history is the updated list.
    """
    conversation_history = conversation_history or []
    conversation_history.append(("user", user_message))
    prompt_tokens = _build_prompt_tokens(conversation_history)
    return prompt_tokens, conversation_history


def _apply_repetition_penalty(logits, generated_ids):
    """
    Applies a repetition penalty to the logits by dividing the scores of tokens
    that have already been generated making them less likely to be repeated.

    Args:
        logits (torch.Tensor): The raw logits tensor of shape (vocab_size,) from
            the model's last position output.
        generated_ids (list): A list of token IDs that have already been generated
            in the current response.

    Returns:
        torch.Tensor: The modified logits tensor with repetition penalty applied
            to previously generated token positions.
    """
    if not generated_ids:
        return logits
    for token_id in set(generated_ids):
        if logits[token_id] > 0:
            logits[token_id] /= inf_repetition_penalty
        else:
            logits[token_id] *= inf_repetition_penalty
    return logits


def _sample_next_token(logits):
    """
    Samples the next token from the logits distribution using temperature scaling
    and top-k filtering for controlled randomness in generation.

    Args:
        logits (torch.Tensor): The raw logits tensor of shape (vocab_size,) from
            the model's output at the last sequence position.

    Returns:
        int: The sampled token ID as a Python integer selected from the filtered
            and temperature-scaled probability distribution.
    """
    logits = logits / inf_temperature
    top_k_values, top_k_indices = torch.topk(logits, inf_top_k)
    probs = torch.softmax(top_k_values, dim=-1)
    sampled_index = torch.multinomial(probs, num_samples=1)
    return top_k_indices[sampled_index].item()


def _decode_single_token(idx, generated_ids):
    """
    Performs a single autoregressive decoding step by running the model forward
    on the current sequence and sampling the next token with penalties applied.

    Args:
        idx (torch.Tensor): Current token sequence tensor of shape (1, T).
        generated_ids (list): List of previously generated token IDs for
            repetition penalty tracking.

    Returns:
        int: The sampled next token ID after applying repetition penalty,
            temperature scaling, and top-k filtering.
    """
    logits = model(idx[:, -block_size:])[0][:, -1, :]
    logits = _apply_repetition_penalty(logits[0], generated_ids)
    return _sample_next_token(logits)


def _is_end_token(token_id):
    """
    Checks whether a generated token ID corresponds to an end-of-conversation or
    end-of-turn delimiter that should terminate response generation.

    Args:
        token_id (int): The token ID to check against known end delimiter strings.

    Returns:
        bool: True if the token represents an end-of-conversation marker or a user
            turn start marker indicating the model is trying to generate a new turn.
    """
    token_text = enc.decode([token_id])
    return "<|end|>" in token_text or "<|user|>" in token_text


def _generate_token_loop(idx, generated_ids):
    """
    Runs the autoregressive token generation loop until an end delimiter is
    produced or the maximum number of new tokens has been reached.

    Args:
        idx (torch.Tensor): Initial token sequence tensor of shape (1, T).
        generated_ids (list): Empty list that will be populated with generated
            token IDs during the loop.

    Returns:
        tuple: A tuple of (idx, generated_ids) where idx is the extended sequence
            tensor and generated_ids contains all newly generated token IDs.
    """
    for _ in range(inf_max_new_tokens):
        next_token = _decode_single_token(idx, generated_ids)
        if _is_end_token(next_token):
            break
        generated_ids.append(next_token)
        idx = torch.cat([idx, torch.tensor([[next_token]], device=device)], dim=1)
    return idx, generated_ids


def _clean_response(generated_ids):
    """
    Decodes generated token IDs into text and removes any residual delimiter
    tokens or formatting artifacts that may have leaked into the output.

    Args:
        generated_ids (list): A list of integer token IDs produced by the
            autoregressive generation loop.

    Returns:
        str: The cleaned response text with all chat format delimiters removed
            and a fallback message if the response is empty.
    """
    response = enc.decode(generated_ids).strip()
    for tag in ["<|user|>", "<|assistant|>", "<|end|>"]:
        response = response.split(tag)[0]
    response = response.strip()
    if not response:
        response = "I'm not sure how to respond to that. Could you rephrase?"
    return response


def generate_response(user_message, conversation_history=None):
    """
    Generates a chatbot response for a given user message using the full
    conversation history for context and autoregressive token decoding.

    Args:
        user_message (str): The user's input text to respond to.
        conversation_history (list): Optional list of (role, text) tuples
            representing previous conversation turns or None for a fresh start.

    Returns:
        str: The model's generated response text cleaned of any delimiters.
    """
    prompt_tokens, conversation_history = _prepare_prompt(
        user_message, conversation_history
    )
    idx = torch.tensor([prompt_tokens[-block_size:]], dtype=torch.long, device=device)
    model.eval()
    with torch.no_grad():
        _, generated_ids = _generate_token_loop(idx, [])
    response = _clean_response(generated_ids)
    conversation_history.append(("assistant", response))
    return response

In [ ]:
# --- Test the chatbot ---
test_prompts = [
    "Hello, who are you?",
    "What is your name?",
    "Tell me about yourself.",
    "What can you help me with?",
    "Who created you?",
]

print("=" * 60)
print("  KGPT Chatbot — Inference Demo")
print("=" * 60)

for prompt in test_prompts:
    print(f"\nYou: {prompt}")
    response = generate_response(prompt)
    print(f"KGPT: {response}")

## Section 6 — Download Weights

Run this cell to copy the model files to Kaggle's output directory. After saving the notebook version, download them from the **Output** tab to use locally with `inference.py`.

In [ ]:
# copy model weight files to Kaggle output directory for download
import shutil

for fname in ["kgpt_pretrained.pt", "kgpt_finetuned.pt", "kgpt_checkpoint.pt"]:
    src = os.path.join(WORK_DIR, fname)
    if os.path.exists(src):
        size_mb = os.path.getsize(src) / (1024**2)
        print(f"{fname} ({size_mb:.0f} MB) -- available in Output tab")
    else:
        print(f"{fname} -- not found")

print("\nSave this notebook version, then download weights from the Output tab.")
print("Place kgpt_finetuned.pt in your local project folder and run:")
print("  python inference.py")